In [ ]:
# GoogLeNet-like model

import os
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow import keras
from tensorflow.keras import layers, Model
from sklearn.metrics import classification_report, confusion_matrix

# 1. Paths

train_dir = "/Users/uyennguyen/MA_DAAN_2025_2026 /DAAN 570/Team Project_570/Data/train"
test_dir = "/Users/uyennguyen/MA_DAAN_2025_2026 /DAAN 570/Team Project_570/Data/test"

# Save artifacts inside the current repo/project folder
repo_root = os.getcwd()
artifacts_dir = os.path.join(repo_root, "artifacts")
os.makedirs(artifacts_dir, exist_ok=True)

print("Repo root:", repo_root)
print("Artifacts folder:", artifacts_dir)

# 2. Load data

img_size = (180, 180)
batch_size = 32

train_data = keras.utils.image_dataset_from_directory(
    train_dir,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=img_size,
    batch_size=batch_size,
    color_mode="grayscale"
)

val_data = keras.utils.image_dataset_from_directory(
    train_dir,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=img_size,
    batch_size=batch_size,
    color_mode="grayscale"
)

test_data = keras.utils.image_dataset_from_directory(
    test_dir,
    image_size=img_size,
    batch_size=batch_size,
    color_mode="grayscale",
    shuffle=False   
)

print("Class names:", train_data.class_names)
print("Number of classes:", len(train_data.class_names))

class_names = train_data.class_names
num_classes = len(class_names)

# 3. Optimize data pipeline

AUTOTUNE = tf.data.AUTOTUNE

train_data = train_data.shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_data = val_data.prefetch(buffer_size=AUTOTUNE)
test_data = test_data.prefetch(buffer_size=AUTOTUNE)

Repo root: /Users/uyennguyen/MA_DAAN_2025_2026 /DAAN 570/Team Project_570/notebooks
Artifacts folder: /Users/uyennguyen/MA_DAAN_2025_2026 /DAAN 570/Team Project_570/notebooks/artifacts
Found 28709 files belonging to 7 classes.
Using 22968 files for training.


2026-03-28 22:12:39.440075: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M4 Max
2026-03-28 22:12:39.440099: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 128.00 GB
2026-03-28 22:12:39.440102: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 53.76 GB
2026-03-28 22:12:39.440115: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-03-28 22:12:39.440123: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


Found 28709 files belonging to 7 classes.
Using 5741 files for validation.
Found 7178 files belonging to 7 classes.
Class names: ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']
Number of classes: 7


In [4]:
# 4. Data augmentation

data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
    layers.RandomContrast(0.1),
], name="data_augmentation")

# 5. Inception block

def inception_block(x, f1, f3_reduce, f3, f5_reduce, f5, fpool):
    branch1 = layers.Conv2D(f1, (1, 1), padding="same", activation="relu")(x)

    branch2 = layers.Conv2D(f3_reduce, (1, 1), padding="same", activation="relu")(x)
    branch2 = layers.Conv2D(f3, (3, 3), padding="same", activation="relu")(branch2)

    branch3 = layers.Conv2D(f5_reduce, (1, 1), padding="same", activation="relu")(x)
    branch3 = layers.Conv2D(f5, (5, 5), padding="same", activation="relu")(branch3)

    branch4 = layers.MaxPooling2D((3, 3), strides=(1, 1), padding="same")(x)
    branch4 = layers.Conv2D(fpool, (1, 1), padding="same", activation="relu")(branch4)

    output = layers.Concatenate(axis=-1)([branch1, branch2, branch3, branch4])
    return output

# 6. Build GoogLeNet-like model

def build_googlenet_like(input_shape=(180, 180, 1), num_classes=7):
    inputs = layers.Input(shape=input_shape)

    x = data_augmentation(inputs)
    x = layers.Rescaling(1.0 / 255.0)(x)

    # Stem
    x = layers.Conv2D(64, (7, 7), strides=2, padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((3, 3), strides=2, padding="same")(x)

    x = layers.Conv2D(64, (1, 1), padding="same", activation="relu")(x)
    x = layers.Conv2D(192, (3, 3), padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((3, 3), strides=2, padding="same")(x)

    # Inception blocks
    x = inception_block(x, f1=64,  f3_reduce=96,  f3=128, f5_reduce=16, f5=32,  fpool=32)
    x = inception_block(x, f1=128, f3_reduce=128, f3=192, f5_reduce=32, f5=96,  fpool=64)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((3, 3), strides=2, padding="same")(x)

    x = inception_block(x, f1=192, f3_reduce=96,  f3=208, f5_reduce=16, f5=48,  fpool=64)
    x = inception_block(x, f1=160, f3_reduce=112, f3=224, f5_reduce=24, f5=64,  fpool=64)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((3, 3), strides=2, padding="same")(x)

    # Head
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.4)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)

    model = Model(inputs, outputs, name="GoogLeNet_like")
    return model

model = build_googlenet_like(
    input_shape=(180, 180, 1),
    num_classes=num_classes
)

model.summary()

# 7. Compile model

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.0001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

# 8. Train model

callbacks = [
    keras.callbacks.ModelCheckpoint(
        os.path.join(artifacts_dir, "best_googlenet_like.keras"),
        monitor="val_accuracy",
        save_best_only=True,
        mode="max"
    ),
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=15,
        restore_best_weights=True
    )
]

history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=40,
    callbacks=callbacks
)

Model: "GoogLeNet_like"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_2       │ (None, 180, 180,  │          0 │ -                 │
│ (InputLayer)        │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ data_augmentation   │ (None, 180, 180,  │          0 │ input_layer_2[0]… │
│ (Sequential)        │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling_1         │ (None, 180, 180,  │          0 │ data_augmentatio… │
│ (Rescaling)         │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_27 (Conv2D)  │ (None, 90, 90,    │      3,200 │ rescaling_1[0][0] │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 90, 90,    │        256 │ conv2d_27[0][0]   │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_8     │ (None, 45, 45,    │          0 │ batch_normalizat… │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_28 (Conv2D)  │ (None, 45, 45,    │      4,160 │ max_pooling2d_8[… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_29 (Conv2D)  │ (None, 45, 45,    │    110,784 │ conv2d_28[0][0]   │
│                     │ 192)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 45, 45,    │        768 │ conv2d_29[0][0]   │
│ (BatchNormalizatio… │ 192)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_9     │ (None, 23, 23,    │          0 │ batch_normalizat… │
│ (MaxPooling2D)      │ 192)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_31 (Conv2D)  │ (None, 23, 23,    │     18,528 │ max_pooling2d_9[… │
│                     │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_33 (Conv2D)  │ (None, 23, 23,    │      3,088 │ max_pooling2d_9[… │
│                     │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_10    │ (None, 23, 23,    │          0 │ max_pooling2d_9[… │
│ (MaxPooling2D)      │ 192)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_30 (Conv2D)  │ (None, 23, 23,    │     12,352 │ max_pooling2d_9[… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_32 (Conv2D)  │ (None, 23, 23,    │    110,720 │ conv2d_31[0][0]   │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_34 (Conv2D)  │ (None, 23, 23,    │     12,832 │ conv2d_33[0][0]   │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_35 (Conv2D)  │ (None, 23, 23,    │      6,176 │ max_pooling2d_10

 Total params: 1,504,495 (5.74 MB)

 Trainable params: 1,501,999 (5.73 MB)

 Non-trainable params: 2,496 (9.75 KB)

Epoch 1/40
718/718 ━━━━━━━━━━━━━━━━━━━━ 117s 155ms/step - accuracy: 0.2527 - loss: 1.9436 - val_accuracy: 0.2933 - val_loss: 1.7207
Epoch 2/40
718/718 ━━━━━━━━━━━━━━━━━━━━ 107s 148ms/step - accuracy: 0.3356 - loss: 1.6654 - val_accuracy: 0.2886 - val_loss: 1.7163
Epoch 3/40
718/718 ━━━━━━━━━━━━━━━━━━━━ 109s 150ms/step - accuracy: 0.3975 - loss: 1.5422 - val_accuracy: 0.4327 - val_loss: 1.4932
Epoch 4/40
718/718 ━━━━━━━━━━━━━━━━━━━━ 109s 150ms/step - accuracy: 0.4319 - loss: 1.4721 - val_accuracy: 0.4342 - val_loss: 1.4988
Epoch 5/40
718/718 ━━━━━━━━━━━━━━━━━━━━ 108s 150ms/step - accuracy: 0.4554 - loss: 1.4099 - val_accuracy: 0.4625 - val_loss: 1.3911
Epoch 6/40
718/718 ━━━━━━━━━━━━━━━━━━━━ 108s 149ms/step - accuracy: 0.4763 - loss: 1.3671 - val_accuracy: 0.5020 - val_loss: 1.3163
Epoch 7/40
718/718 ━━━━━━━━━━━━━━━━━━━━ 109s 151ms/step - accuracy: 0.4980 - loss: 1.3121 - val_accuracy: 0.5177 - val_loss: 1.2890
Epoch 8/40
718/718 ━━━━━━━━━━━━━━━━━━━━ 110s 152ms/step - accuracy: 0.5100 -

In [5]:
# 9. Evaluate model

test_loss, test_acc = model.evaluate(test_data)
print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_acc:.4f}")

# 10. Predictions

y_true = np.concatenate([y.numpy() for x, y in test_data], axis=0)
y_pred_probs = model.predict(test_data)
y_pred = np.argmax(y_pred_probs, axis=1)


# 11. Reports

report_text = classification_report(
    y_true,
    y_pred,
    target_names=class_names,
    digits=4
)

cm = confusion_matrix(y_true, y_pred)

print("\nClassification Report:\n")
print(report_text)

print("\nConfusion Matrix:\n")
print(cm)

# 12. Visualizations

desired_accuracy = 0.65
train_acc_history = history.history["accuracy"]
val_acc_history = history.history["val_accuracy"]
epochs = range(1, len(train_acc_history) + 1)

# Plot 1: Training accuracy vs test accuracy vs desired accuracy
plt.figure(figsize=(10, 6))
plt.plot(epochs, train_acc_history, linewidth=2, label="Training accuracy")
plt.axhline(y=test_acc, linestyle="--", linewidth=2, label=f"Test accuracy = {test_acc:.4f}")
plt.axhline(y=desired_accuracy, linestyle="-", linewidth=2, label="Desired accuracy = 0.65")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training Accuracy vs Test Accuracy")
plt.ylim(0, 1.0)
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig(os.path.join(artifacts_dir, "accuracy_comparison.png"), dpi=300, bbox_inches="tight")
plt.close()

# Plot 2: Training accuracy vs validation accuracy vs desired accuracy
plt.figure(figsize=(10, 6))
plt.plot(epochs, train_acc_history, linewidth=2, label="Training accuracy")
plt.plot(epochs, val_acc_history, linewidth=2, label="Validation accuracy")
plt.axhline(y=desired_accuracy, linestyle="-", linewidth=2, label="Desired accuracy = 0.65")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training Accuracy vs Validation Accuracy")
plt.ylim(0, 1.0)
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig(os.path.join(artifacts_dir, "train_vs_validation_accuracy.png"), dpi=300, bbox_inches="tight")
plt.close()

# 13. Save text artifacts

with open(os.path.join(artifacts_dir, "classification_report.txt"), "w", encoding="utf-8") as f:
    f.write(report_text)

with open(os.path.join(artifacts_dir, "confusion_matrix.txt"), "w", encoding="utf-8") as f:
    f.write("Confusion Matrix\n")
    f.write(np.array2string(cm))

with open(os.path.join(artifacts_dir, "test_metrics.txt"), "w", encoding="utf-8") as f:
    f.write(f"Test loss: {test_loss:.4f}\n")
    f.write(f"Test accuracy: {test_acc:.4f}\n")
    f.write(f"Desired accuracy: {desired_accuracy:.2f}\n")

with open(os.path.join(artifacts_dir, "class_names.txt"), "w", encoding="utf-8") as f:
    for idx, class_name in enumerate(class_names):
        f.write(f"{idx}: {class_name}\n")


# 14. Save training history

history_file = os.path.join(artifacts_dir, "training_history.csv")
with open(history_file, "w", encoding="utf-8") as f:
    f.write("epoch,accuracy,val_accuracy,loss,val_loss\n")
    for i in range(len(history.history["accuracy"])):
        f.write(
            f"{i+1},"
            f"{history.history['accuracy'][i]:.6f},"
            f"{history.history['val_accuracy'][i]:.6f},"
            f"{history.history['loss'][i]:.6f},"
            f"{history.history['val_loss'][i]:.6f}\n"
        )

# 15. Done

print(f"\nArtifacts saved in folder: {artifacts_dir}")
print("Files created:")
print("- best_googlenet_like.keras")
print("- classification_report.txt")
print("- confusion_matrix.txt")
print("- test_metrics.txt")
print("- class_names.txt")
print("- training_history.csv")
print("- accuracy_comparison.png")
print("- train_vs_validation_accuracy.png")

225/225 ━━━━━━━━━━━━━━━━━━━━ 3s 12ms/step - accuracy: 0.6151 - loss: 1.0263
Test loss: 1.0263
Test accuracy: 0.6151


2026-03-28 23:28:35.245323: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


225/225 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step

Classification Report:

              precision    recall  f1-score   support

       angry     0.4665    0.6534    0.5443       958
     disgust     0.8500    0.1532    0.2595       111
        fear     0.5246    0.2607    0.3483      1024
       happy     0.8874    0.7999    0.8414      1774
     neutral     0.5099    0.7315    0.6009      1233
         sad     0.5556    0.3809    0.4520      1247
    surprise     0.6541    0.8532    0.7405       831

    accuracy                         0.6151      7178
   macro avg     0.6354    0.5476    0.5410      7178
weighted avg     0.6294    0.6151    0.6018      7178


Confusion Matrix:

[[ 626    2   45   32  137   73   43]
 [  56   17    9    3    8   11    7]
 [ 218    1  267   30  172  159  177]
 [  86    0   23 1419  154   22   70]
 [ 100    0   42   47  902  107   35]
 [ 220    0   95   41  373  475   43]
 [  36    0   28   27   23    8  709]]

Artifacts saved in folder: /Users/uyennguyen/MA_D